# HDB Flat Recommender — Mockup
### DSE3101 | Quiz-driven amenity weighting + price value scoring

This notebook runs the full recommender pipeline interactively.  
Run each cell in order. Cells that require user input will prompt you inline.

---
**Stages:**
1. Hard constraint filtering (budget, bedrooms, town)
2. Quiz → amenity scoring → normalisation → tie-breaking → ranking
3. Rank-Sum weights → listing scoring → top 5 results


In [ ]:
import pandas as pd
import numpy as np
import sys


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/dse3101/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## ⚙️ Cell 2 — Configuration
All tunable parameters. Edit paths before running.

In [ ]:
# ── FILE PATHS ──────────────────────────────────────────────────
# Update these to your local or Drive paths
LISTINGS_PATH = BASE_PATH + 'active_listings.xlsx'
AMENITY_PATH  = BASE_PATH + 'hdb_amenity_table_full.csv'

# ── DECAY PARAMETERS ─────────────────────────────────────────────
# TAU = characteristic distance: score drops to ~37% at TAU metres
TAU_DEFAULT = 500.0   # metres

# ── SCORING ──────────────────────────────────────────────────────
VALUE_CLIP   = 0.10   # clip pct_gap to ±10% before normalising value score
SHORTLIST_N  = 30     # listings to shortlist after Stage 3A
TOP_N        = 5      # final listings to return

# ── TIE-BREAK ────────────────────────────────────────────────────
TIE_THRESHOLD       = 0.01   # normalised score gap that triggers a tie-break question
SECONDARY_THRESHOLD = 2      # offer optional secondary amenities if active count <= this

# ── CLUSTER FLAG ─────────────────────────────────────────────────
CLUSTER_THRESHOLD_M = 300    # flag if std dev of listing coords implies spread < this (metres)

# ── AMENITIES ────────────────────────────────────────────────────
AMENITIES = ['train', 'bus', 'hawker', 'supermarket', 'mall', 'school', 'polyclinic']

# ── AMENITY DISPLAY LABELS ────────────────────────────────────────
AMENITY_LABELS = {
    'train':       'Transport  — MRT / train',
    'bus':         'Transport  — Bus',
    'hawker':      'Food       — Hawker centres',
    'mall':        'Shopping   — Malls',
    'supermarket': 'Groceries  — Supermarkets',
    'school':      'Education  — Primary schools',
    'polyclinic':  'Healthcare — Polyclinics',
}

print("✓ Config loaded")
print(f"  Listings path : {LISTINGS_PATH}")
print(f"  Amenity path  : {AMENITY_PATH}")
print(f"  TAU           : {TAU_DEFAULT}m")
print(f"  Value clip    : ±{VALUE_CLIP*100:.0f}%")
print(f"  Amenities     : {AMENITIES}")


✓ Config loaded
  Listings path : /content/drive/MyDrive/dse3101/active_listings.xlsx
  Amenity path  : /content/drive/MyDrive/dse3101/hdb_amenity_table_full.csv
  TAU           : 500.0m
  Value clip    : ±10%
  Amenities     : ['train', 'bus', 'hawker', 'supermarket', 'mall', 'school', 'polyclinic']


## 🔧 Cell 3 — Helper Functions

In [ ]:
def ask(prompt, options):
    """Present lettered options, return chosen key."""
    print(f"\n{prompt}")
    keys = list(options.keys())
    for i, k in enumerate(keys):
        print(f"  [{chr(65+i)}] {options[k]}")
    while True:
        raw = input("  Your choice: ").strip().upper()
        if raw and len(raw) > 0 and ord(raw[0]) - 65 < len(keys) and ord(raw[0]) >= 65:
            idx = ord(raw[0]) - 65
            chosen = keys[idx]
            print(f"  → {options[chosen]}")
            return chosen
        print("  Please enter a valid letter.")


def haversine_vec(lat1, lon1, lats2, lons2):
    """Vectorised Haversine: one point vs array. Returns distances in metres."""
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lats2)
    dphi = np.radians(lats2 - lat1)
    dlam = np.radians(lons2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))


def exp_decay(dist_m, tau):
    """Exponential decay: score = exp(-dist / tau). Score = 0.37 at dist = tau."""
    return np.exp(-np.asarray(dist_m) / tau)


def rank_sum_weights(ranking):
    """Convert ordered amenity list to Rank-Sum weights dict."""
    n = len(ranking)
    total = n * (n + 1) / 2
    return {amenity: (n - i) / total for i, amenity in enumerate(ranking)}


def print_divider(title=""):
    w = 62
    if title:
        pad = (w - len(title) - 2) // 2
        print("\n" + "─"*pad + f" {title} " + "─"*pad)
    else:
        print("\n" + "─"*w)


print("✓ Helper functions loaded")


✓ Helper functions loaded


## 📂 Cell 4 — Load Data

In [ ]:
def load_data():
    listings = pd.read_excel(LISTINGS_PATH)
    listings = listings.rename(columns={
        'price/value':      'asking_price',
        'floor_area (sqft)':'floor_area_sqft',
        'ï»¿bathrooms':     'bathrooms',
    })
    listings = listings[listings['planning_area'] != 'ERROR'].copy()
    listings = listings.dropna(subset=['latitude', 'longitude'])
    listings = listings.drop_duplicates(subset=['full_address']).reset_index(drop=True)

    # Simulate predicted price: asking ± ~5% noise, slight downward bias
    # Replace this with real model predictions when available (IMPORTANT)
    np.random.seed(42)
    noise = np.random.normal(-0.03, 0.05, len(listings))
    listings['predicted_price']  = (listings['asking_price'] * (1 + noise)).round(-3)
    listings['value_delta_pct']  = (
        (listings['predicted_price'] - listings['asking_price'])
        / listings['predicted_price'] * 100
    ).round(1)
    listings['value_delta_abs']  = (
        listings['predicted_price'] - listings['asking_price']
    ).round(-3).astype(int)

    amenity_df = pd.read_csv(AMENITY_PATH)
    return listings, amenity_df


listings, amenity_df = load_data()
print(f"✓ Loaded {len(listings)} listings | {len(amenity_df)} HDB blocks with amenity data")
print(f"  Listing columns : {listings.columns.tolist()}")
print(f"  Towns in pool   : {sorted(listings['planning_area'].unique().tolist())}")


✓ Loaded 809 listings | 10743 HDB blocks with amenity data
  Listing columns : ['agency/name', 'Built year', 'floor_area_sqft', 'full_address', 'mrt/nearby_text', 'posted_on', 'asking_price', 'price_per_area/locale_string_value', 'property/id', 'latitude', 'longitude', 'planning_area', 'bathrooms', 'bedrooms', 'predicted_price', 'value_delta_pct', 'value_delta_abs']
  Towns in pool   : ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG', 'MARINE PARADE', 'NOVENA', 'OUTRAM', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'ROCHOR', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']


## 📐 Cell 5 — Amenity Scoring Functions

In [ ]:
def match_to_amenity_table(listings, amenity_df):
    """Match each listing to its nearest HDB block by lat/lon (vectorised)."""
    a_lats = amenity_df['lat'].values
    a_lons = amenity_df['lon'].values
    indices = []
    for _, row in listings.iterrows():
        if pd.isna(row['latitude']) or pd.isna(row['longitude']):
            indices.append(None)
        else:
            dists = haversine_vec(row['latitude'], row['longitude'], a_lats, a_lons)
            indices.append(int(np.argmin(dists)))
    return indices


def get_amenity_scores(amenity_row, tau):
    """
    For a given HDB block row, compute accessibility score per amenity.
    Score = mean(exp(-dist_i / tau)) across top 3 nearest locations.
    TAU is passed in per-listing based on life stage.
    """
    scores = {}
    for amenity in AMENITIES:
        dists = []
        for n in [1, 2, 3]:
            col = f'{amenity}_{n}_dist_m'
            if col in amenity_row.index and pd.notna(amenity_row[col]):
                dists.append(amenity_row[col])
        scores[amenity] = float(np.mean(exp_decay(dists, tau))) if dists else 0.0
    return scores


def check_geographic_clustering(filtered):
    """
    Check if filtered listings are geographically clustered.
    Returns (is_clustered: bool, spread_m: float).
    Uses std dev of lat/lon converted to approximate metres.
    """
    if len(filtered) <= 1:
        return True, 0.0
    lat_std = filtered['latitude'].std()
    lon_std = filtered['longitude'].std()
    # 1 degree lat ≈ 111,000m; 1 degree lon ≈ 111,000 * cos(lat)
    avg_lat = filtered['latitude'].mean()
    spread_m = ((lat_std * 111000)**2 + (lon_std * 111000 * np.cos(np.radians(avg_lat)))**2)**0.5
    return spread_m < CLUSTER_THRESHOLD_M, round(spread_m)


print("✓ Amenity scoring functions loaded")


✓ Amenity scoring functions loaded


## ❓ Cell 6 — Master Question Bank
10 lifestyle questions, each option mapped to one amenity.
Backend filters this bank at runtime based on the user's selected amenities.
Questions with fewer than 2 valid options after filtering are dropped entirely.


In [ ]:
# Master question bank — 10 lifestyle questions
# Each option maps to one amenity key (or None for 'no preference').
# The 'no_pref' option is always appended automatically at runtime — do not add it here.
# Questions with fewer than 2 valid amenity options after filtering are skipped.

QUESTION_BANK = [
    {
        'id': 'q1',
        'text': 'Which of these feels most like your usual evening?',
        'options': [
            {'id': 'q1_a', 'label': 'Whipping something up at home',         'amenity': 'supermarket'},
            {'id': 'q1_b', 'label': 'Grabbing a quick meal nearby',           'amenity': 'hawker'},
            {'id': 'q1_c', 'label': 'Heading to the mall for dinner and errands', 'amenity': 'mall'},
        ],
    },
    {
        'id': 'q2',
        'text': 'What makes getting home feel easiest?',
        'options': [
            {'id': 'q2_a', 'label': 'A fast MRT ride',           'amenity': 'train'},
            {'id': 'q2_b', 'label': 'A bus stop close to home',  'amenity': 'bus'},
        ],
    },
    {
        'id': 'q3',
        'text': 'How do you usually stock up on food?',
        'options': [
            {'id': 'q3_a', 'label': 'Frequent supermarket top-ups',        'amenity': 'supermarket'},
            {'id': 'q3_b', 'label': 'I mostly eat out or takeaway',         'amenity': 'hawker'},
            {'id': 'q3_c', 'label': 'I usually buy food while at the mall', 'amenity': 'mall'},
        ],
    },
    {
        'id': 'q4',
        'text': 'On a busy weekday, which convenience matters most?',
        'options': [
            {'id': 'q4_a', 'label': 'Bus stop downstairs',                      'amenity': 'bus'},
            {'id': 'q4_b', 'label': 'Mall nearby for one-stop convenience',      'amenity': 'mall'},
            {'id': 'q4_c', 'label': 'Polyclinic nearby for peace of mind',       'amenity': 'polyclinic'},
        ],
    },
    {
        'id': 'q5',
        'text': 'Which of these would improve your daily routine the most?',
        'options': [
            {'id': 'q5_a', 'label': 'Shorter MRT commute',              'amenity': 'train'},
            {'id': 'q5_b', 'label': 'Affordable food options nearby',   'amenity': 'hawker'},
            {'id': 'q5_c', 'label': 'Groceries within easy reach',      'amenity': 'supermarket'},
        ],
    },
    {
        'id': 'q6',
        'text': 'What sounds most like your weekend?',
        'options': [
            {'id': 'q6_a', 'label': 'Walking around the mall',      'amenity': 'mall'},
            {'id': 'q6_b', 'label': 'Meal prep and grocery run',    'amenity': 'supermarket'},
            {'id': 'q6_c', 'label': 'Trying nearby food spots',     'amenity': 'hawker'},
        ],
    },
    {
        'id': 'q7',
        'text': 'Which would make you feel more settled in a neighbourhood?',
        'options': [
            {'id': 'q7_a', 'label': 'Healthcare nearby',                      'amenity': 'polyclinic'},
            {'id': 'q7_b', 'label': 'School access matters for my household', 'amenity': 'school'},
            {'id': 'q7_c', 'label': 'Good transport connectivity',            'amenity': 'train'},
        ],
    },
    {
        'id': 'q8',
        'text': 'If you need to pop out quickly, what would you most want nearby?',
        'options': [
            {'id': 'q8_a', 'label': 'Bus stop',                          'amenity': 'bus'},
            {'id': 'q8_b', 'label': 'Mall with many things in one place', 'amenity': 'mall'},
            {'id': 'q8_c', 'label': 'Polyclinic',                        'amenity': 'polyclinic'},
        ],
    },
    {
        'id': 'q9',
        'text': 'Which statement sounds most like you?',
        'options': [
            {'id': 'q9_a', 'label': 'School access is important now or soon', 'amenity': 'school'},
            {'id': 'q9_b', 'label': 'Food convenience matters more',          'amenity': 'hawker'},
            {'id': 'q9_c', 'label': 'Transport convenience matters more',     'amenity': 'train'},
        ],
    },
    {
        'id': 'q10',
        'text': 'Which trade-off would you choose?',
        'options': [
            {'id': 'q10_a', 'label': 'Near MRT over more food options',  'amenity': 'train'},
            {'id': 'q10_b', 'label': 'Near hawker over faster transport', 'amenity': 'hawker'},
        ],
    },
]

print(f"✓ Question bank loaded: {len(QUESTION_BANK)} questions")


✓ Question bank loaded: 10 questions


## 🔍 Cell 7 — Stage 1: Hard Constraint Filtering
Filters the listing pool by budget, minimum bedrooms, and preferred town(s).  
If no results are found, shows a budget hint and nearby towns with supply.


In [ ]:
def ask_multiselect(prompt, options):
    """Multi-select input: user enters letters like 'ACE' or 'A C E'. Returns list of chosen keys."""
    print(f"\n{prompt}")
    keys = list(options.keys())
    for i, k in enumerate(keys):
        print(f"  [{chr(65+i)}] {options[k]}")
    print("  (Select one or more, e.g. 'A', 'AC', 'A C E')")
    while True:
        raw    = input("  Your choices: ").strip().upper().replace(" ", "")
        chosen, seen, valid = [], set(), True
        for ch in raw:
            idx = ord(ch) - 65
            if 0 <= idx < len(keys) and keys[idx] not in seen:
                chosen.append(keys[idx])
                seen.add(keys[idx])
            elif not (0 <= idx < len(keys)):
                valid = False
        if chosen and valid:
            for k in chosen:
                print(f"  ✓ {options[k]}")
            return chosen
        print("  Please enter valid letters (at least one).")

def stage1_filter(listings):
    print_divider("STAGE 1: Hard Constraints")

    budget    = int(input("\n  Max budget — asking price (SGD, e.g. 700000): $").strip())

    room_types = ask_multiselect("  Number of bedrooms (select all acceptable):", {
        '1': '1-room',
        '2': '2-room',
        '3': '3-room',
        '4': '4-room',
        '5': '5-room',
    })
    room_types = [int(r) for r in room_types]

    town_input = input("  Preferred town(s) — comma separated, or ENTER for any: ").strip().upper()
    preferred_towns = [t.strip() for t in town_input.split(',')] if town_input else []

    filtered = listings[
        (listings['asking_price'] <= budget) &
        (listings['bedrooms'].isin(room_types))
    ].copy()
    if preferred_towns:
        filtered = filtered[filtered['planning_area'].isin(preferred_towns)]

    print(f"\n  Asking price ≤ ${budget:,} | Bedrooms: {room_types}"
          + (f" | Towns: {preferred_towns}" if preferred_towns else " | Any town"))
    print(f"\n  ✓ {len(filtered)} listings pass hard filters (from {len(listings)} total)")

    if len(filtered) == 0:
        print("\n  ⚠️  No listings found matching those constraints.")

        # Budget hint — what does this town+bedroom combo actually cost?
        if preferred_towns:
            town_match = listings[
                (listings['bedrooms'].isin(room_types)) &
                (listings['planning_area'].isin(preferred_towns))
            ]
            if len(town_match) > 0:
                min_p = int(town_match['asking_price'].min())
                med_p = int(town_match['asking_price'].median())
                gap   = min_p - budget
                print(f"\n  Budget hint for {preferred_towns[0].title()} with {room_types}-bed:")
                print(f"    Cheapest asking price : ${min_p:,}  (you are ${gap:,} short)")
                print(f"    Median asking price   : ${med_p:,}")
            else:
                print(f"\n  No {room_types}-bed listings in {preferred_towns} at any price in current pool.")

        # Nearby towns with supply — sorted by distance from chosen town
        budget_match = listings[
            (listings['asking_price'] <= budget) &
            (listings['bedrooms'].isin(room_types))
        ]
        if len(budget_match) > 0 and preferred_towns:
            town_centroids = listings.groupby('planning_area')[['latitude','longitude']].mean()
            chosen = None
            for t in preferred_towns:
                if t in town_centroids.index:
                    chosen = town_centroids.loc[t]
                    break

            towns = budget_match['planning_area'].unique().tolist()
            if chosen is not None:
                towns = sorted(towns, key=lambda t: haversine_vec(
                    chosen['latitude'], chosen['longitude'],
                    np.array([town_centroids.loc[t,'latitude']]),
                    np.array([town_centroids.loc[t,'longitude']])
                )[0] if t in town_centroids.index else float('inf'))

            print(f"\n  Nearby towns with {room_types}-bed listings within ${budget:,} (asking price):")
            for town in towns[:8]:
                sub   = budget_match[budget_match['planning_area'] == town]
                min_p = int(sub['asking_price'].min())
                count = len(sub)
                print(f"    {town.title():<24} {count:>3} listing(s)   from ${min_p:,}")

        raise SystemExit("Re-run Cell 7 with relaxed constraints.")

    elif len(filtered) < 10:
        print(f"\n  ⚠️  WARNING: Only {len(filtered)} listings — results may be limited. Consider relaxing constraints.")

    # Geographic clustering check
    is_clustered, spread_m = check_geographic_clustering(filtered)
    if is_clustered:
        print(f"\n  📍 Note: Filtered listings are geographically clustered (~{spread_m}m spread).")
        print("     Amenity scores will be very similar — price value will be the main differentiator.")

    return filtered, {'budget': budget, 'room_types': room_types,
                      'preferred_towns': preferred_towns, 'is_clustered': is_clustered}


filtered, constraints = stage1_filter(listings)



───────────────── STAGE 1: Hard Constraints ─────────────────

  Max budget — asking price (SGD, e.g. 700000): $700000

  Number of bedrooms (select all acceptable):
  [A] 1-room
  [B] 2-room
  [C] 3-room
  [D] 4-room
  [E] 5-room
  (Select one or more, e.g. 'A', 'AC', 'A C E')
  Your choices: B C D E
  ✓ 2-room
  ✓ 3-room
  ✓ 4-room
  ✓ 5-room
  Preferred town(s) — comma separated, or ENTER for any: 

  Asking price ≤ $700,000 | Bedrooms: [2, 3, 4, 5] | Any town

  ✓ 417 listings pass hard filters (from 809 total)


## 🧩 Cell 8 — Stage 2: Amenity Selection + Lifestyle Quiz
Stage 2A: user picks which amenity categories matter to them (multiselect).  
Stage 2B: backend filters the question bank to only questions relevant to selected amenities,
then runs the quiz. Each chosen answer adds +1 to its linked amenity.  
Scores are normalised so they sum to 1 — these become the final preference weights.


In [ ]:
def build_quiz(selected_amenities):
    """
    Filter QUESTION_BANK to only questions relevant to the user's selected amenities.

    Rules (per spec):
      1. Remove any option whose amenity is not in selected_amenities.
      2. Discard the whole question if fewer than 2 valid amenity options remain.
      3. Append a 'No preference' option (amenity=None) to every kept question.
    """
    selected = set(selected_amenities)
    filtered = []
    for q in QUESTION_BANK:
        valid_opts = [o for o in q['options'] if o['amenity'] in selected]
        if len(valid_opts) < 2:
            continue
        kept = dict(q)
        kept['options'] = valid_opts + [{'id': f"{q['id']}_np", 'label': 'No preference', 'amenity': None}]
        filtered.append(kept)
    return filtered


def compute_max_scores(quiz_questions, selected_amenities, base_score=1):
    """
    Maximum possible raw score for each amenity:
      base_score for being selected
      +1 for every quiz question where that amenity appears as a valid option
    """
    max_scores = {a: base_score for a in selected_amenities}
    for q in quiz_questions:
        for o in q['options']:
            amenity = o.get('amenity')
            if amenity in max_scores:
                max_scores[amenity] += 1
    return max_scores


def run_quiz(quiz_questions, selected_amenities, base_score=1):
    """
    Present each filtered question and collect responses.
    Each selected amenity starts with +base_score.
    Each quiz answer adds a further +1 to its linked amenity.
    'No preference' adds 0.
    Returns raw_scores dict.
    """
    print_divider("STAGE 2B: Lifestyle Quiz")
    print(f"  {len(quiz_questions)} questions based on your selected amenities\n")

    raw_scores = {a: base_score for a in selected_amenities}

    for i, q in enumerate(quiz_questions, 1):
        options = {o['id']: o['label'] for o in q['options']}
        chosen_id = ask(f"{i}. {q['text']}", options)

        chosen_opt = next(o for o in q['options'] if o['id'] == chosen_id)
        amenity = chosen_opt['amenity']
        if amenity is not None:
            raw_scores[amenity] = raw_scores.get(amenity, 0) + 1
            print(f"  → {amenity} +1")

    return raw_scores


def normalise_to_weights(raw_scores, max_scores, selected_amenities):
    """
    Fair normalisation in 2 steps:
      1. Convert each amenity to an attainment score = raw / max_possible.
      2. Renormalise attainment scores so final weights sum to 1.

    This avoids bias when some amenities appear in more quiz questions than others.
    """
    attainment = {}
    for a in selected_amenities:
        max_score = max_scores.get(a, 0)
        raw_score = raw_scores.get(a, 0)
        attainment[a] = (raw_score / max_score) if max_score > 0 else 0.0

    total_attainment = sum(attainment.values())
    if total_attainment == 0:
        n = len(selected_amenities)
        return attainment, {a: round(1 / n, 4) for a in selected_amenities}

    weights = {a: round(attainment[a] / total_attainment, 4) for a in selected_amenities}
    attainment = {a: round(attainment[a], 4) for a in selected_amenities}
    return attainment, weights


# ── Stage 2A: amenity selection ───────────────────────────────────────────────
print_divider("STAGE 2A: What Do You Want Near Your Home?")
print("\n  Select the amenity categories that matter to you.")
print("  Only your chosen amenities will appear in quiz questions and scoring.\n")

selected_amenities = ask_multiselect(
    "Which of these do you want near your home? (select all that apply)",
    AMENITY_LABELS,
)
print(f"\n  ✓ Selected: {selected_amenities}")

# ── Stage 2B: filtered quiz ───────────────────────────────────────────────────
quiz_questions = build_quiz(selected_amenities)
max_scores     = compute_max_scores(quiz_questions, selected_amenities, base_score=1)
raw_scores     = run_quiz(quiz_questions, selected_amenities, base_score=1)

# ── Fair normalisation to weights ─────────────────────────────────────────────
attainment_scores, normalised_weights = normalise_to_weights(
    raw_scores, max_scores, selected_amenities
)

print(f"\n  Quiz score summary:")
print(f"  {'Amenity':<14} {'Raw':>6} {'Max':>6} {'Raw/Max':>10} {'Weight':>10}")
print(f"  {'─'*14} {'─'*6} {'─'*6} {'─'*10} {'─'*10}")
for a in selected_amenities:
    print(
        f"  {a:<14} {raw_scores.get(a, 0):>6} {max_scores.get(a, 0):>6} "
        f"{attainment_scores[a]:>10.4f} {normalised_weights[a]:>10.4f}"
    )


───────── STAGE 2A: What Do You Want Near Your Home? ─────────

  Select the amenity categories that matter to you.
  Only your chosen amenities will appear in quiz questions and scoring.


Which of these do you want near your home? (select all that apply)
  [A] Transport  — MRT / train
  [B] Transport  — Bus
  [C] Food       — Hawker centres
  [D] Shopping   — Malls
  [E] Groceries  — Supermarkets
  [F] Education  — Primary schools
  [G] Healthcare — Polyclinics
  (Select one or more, e.g. 'A', 'AC', 'A C E')
  Your choices: A B C D E F G
  ✓ Transport  — MRT / train
  ✓ Transport  — Bus
  ✓ Food       — Hawker centres
  ✓ Shopping   — Malls
  ✓ Groceries  — Supermarkets
  ✓ Education  — Primary schools
  ✓ Healthcare — Polyclinics

  ✓ Selected: ['train', 'bus', 'hawker', 'mall', 'supermarket', 'school', 'polyclinic']

────────────────── STAGE 2B: Lifestyle Quiz ──────────────────
  10 questions based on your selected amenities


1. Which of these feels most like your usual evening?

## 📊 Cell 9 — Build Initial Ranking from Quiz Weights
Sorts the user's selected amenities by their normalised quiz weight (descending).  
These weights directly become the preference ranking fed into Rank-Sum.


In [ ]:
def build_initial_ranking(normalised_weights):
    """
    Sort selected amenities by normalised quiz weight descending.
    Amenities with weight 0 are excluded (inactive).
    Returns (active_ranking, inactive_list).
    """
    active   = [a for a in normalised_weights if normalised_weights[a] > 0]
    inactive = [a for a in normalised_weights if normalised_weights[a] == 0]
    active_ranking = sorted(active, key=lambda a: normalised_weights[a], reverse=True)
    return active_ranking, inactive


init_ranking, inactive = build_initial_ranking(normalised_weights)

print_divider("Initial Suggested Ranking")
print("\n  Active amenities (will be included in scoring):")
for i, a in enumerate(init_ranking, 1):
    bar = '█' * int(normalised_weights[a] * 30)
    print(f"    {i}. {a:<14}  weight: {normalised_weights[a]:.4f}  {bar}")
if inactive:
    print(f"\n  Inactive (score 0 — no quiz answers for these): {', '.join(inactive)}")



───────────────── Initial Suggested Ranking ─────────────────

  Active amenities (will be included in scoring):
    1. bus             weight: 0.2049  ██████
    2. train           weight: 0.1822  █████
    3. school          weight: 0.1822  █████
    4. supermarket     weight: 0.1640  ████
    5. polyclinic      weight: 0.1366  ████
    6. mall            weight: 0.0911  ██
    7. hawker          weight: 0.0390  █


## 🔀 Cell 10 — Stage 2C: Tie-Break Questions
For adjacent pairs in the ranking that share the same normalised weight,
a targeted this-or-that question is asked to resolve the tie.


In [ ]:
def slider_tiebreak(a1, a2):
    """
    Present a text slider between two tied amenities.
    Range: -5 (strongly prefer a1) to +5 (strongly prefer a2). Default 0 = no preference.

    Returns an integer in [-5, +5].
    """
    label_a1 = AMENITY_LABELS[a1]
    label_a2 = AMENITY_LABELS[a2]

    def render(val):
        track = ['─'] * 11
        pos   = val + 5
        track[pos] = '●'
        bar = ''.join(track)
        return f"  {label_a1}  ◄ {bar} ►  {label_a2}  (value: {val:+d})"

    print(f"\n  How much do you prefer one over the other?")
    print(f"  Enter a number from -5 to +5:")
    print(f"    -5 = strongly prefer {label_a1}")
    print(f"     0 = no preference (keep current order)")
    print(f"    +5 = strongly prefer {label_a2}")
    print()
    print(render(0))

    while True:
        raw = input("\n  Your value (-5 to +5): ").strip()
        try:
            val = int(raw)
            if -5 <= val <= 5:
                print(render(val))
                return val
        except ValueError:
            pass
        print("  Please enter a whole number between -5 and +5.")


def find_tied_pairs(ranking, weights, threshold=TIE_THRESHOLD):
    """
    Find adjacent pairs whose weights are genuinely close.
    Use a small threshold; otherwise too many non-tied pairs get sent to tie-break.
    """
    tied = []
    for i in range(len(ranking) - 1):
        a1, a2 = ranking[i], ranking[i+1]
        if abs(weights[a1] - weights[a2]) <= threshold:
            tied.append((a1, a2))
    return tied


def renormalise_weights(weights):
    total = sum(max(v, 0) for v in weights.values())
    if total == 0:
        return weights
    for k in list(weights.keys()):
        weights[k] = round(max(weights[k], 0) / total, 4)
    return weights


def run_tiebreakers(ranking, weights, tied_pairs):
    """
    Resolve only near-ties using a slider.

    Important:
    - This step is NOT the source of the original quiz weights.
    - It only nudges very close items for ordering when the user says one matters more.
    - After nudging, weights are renormalised and the ranking is re-sorted so the
      displayed order always matches the displayed weights.
    """
    NUDGE_SCALE = TIE_THRESHOLD + 0.005

    print_divider("STAGE 2C: Tie-Break")

    if not tied_pairs:
        print("\n  No close ties detected — skipping tie-break.")
        return sorted(list(ranking), key=lambda a: weights[a], reverse=True)

    print(f"\n  Detected {len(tied_pairs)} close tie(s) — use the slider to break each one.\n")

    ranking = list(ranking)
    for a1, a2 in tied_pairs:
        print(f"  Tied: {AMENITY_LABELS[a1]}  vs  {AMENITY_LABELS[a2]}")
        val = slider_tiebreak(a1, a2)

        if val == 0:
            print("  → No preference — order kept as is")
            continue

        nudge  = (abs(val) / 5) * NUDGE_SCALE
        winner = a1 if val < 0 else a2
        loser  = a2 if val < 0 else a1

        weights[winner] = weights.get(winner, 0) + nudge
        weights[loser]  = max(weights.get(loser, 0) - nudge, 0)

        print(f"  → {AMENITY_LABELS[winner]} preferred over {AMENITY_LABELS[loser]}")

    renormalise_weights(weights)
    ranking = sorted(ranking, key=lambda a: weights[a], reverse=True)
    return ranking


tied_pairs   = find_tied_pairs(init_ranking, normalised_weights)
post_tb_rank = run_tiebreakers(init_ranking, normalised_weights, tied_pairs)

print_divider("Ranking After Tie-Breaks")
print("\n  Current amenity ranking:")
for i, a in enumerate(post_tb_rank, 1):
    print(f"    {i}. {a:<14}  weight: {normalised_weights[a]:.4f}")


──────────────────── STAGE 2C: Tie-Break ────────────────────

  Detected 1 close tie(s) — use the slider to break each one.

  Tied: Transport  — MRT / train  vs  Education  — Primary schools

  How much do you prefer one over the other?
  Enter a number from -5 to +5:
    -5 = strongly prefer Transport  — MRT / train
     0 = no preference (keep current order)
    +5 = strongly prefer Education  — Primary schools

  Transport  — MRT / train  ◄ ─────●───── ►  Education  — Primary schools  (value: +0)

  Your value (-5 to +5): -5
  Transport  — MRT / train  ◄ ●────────── ►  Education  — Primary schools  (value: -5)
  → Transport  — MRT / train preferred over Education  — Primary schools

────────────────── Ranking After Tie-Breaks ──────────────────

  Current amenity ranking:
    1. bus             weight: 0.2049
    2. train           weight: 0.1972
    3. school          weight: 0.1672
    4. supermarket     weight: 0.1640
    5. polyclinic      weight: 0.1366
    6. mall          

## ✏️ Cell 11 — Stage 3: Confirm / Adjust Ranking
Review the quiz-suggested ranking and make any final adjustments.  
**Commands:**  
- `swap 2 4` — swap positions 2 and 4  
- `move 3 5` — move item at position 3 to position 5, shifting others  
- Press ENTER to confirm


In [ ]:
def user_adjust_ranking(ranking, suggested_weights):
    print_divider("STAGE 3: Confirm / Adjust Ranking")

    print("\n  Suggested amenity ranking based on your quiz answers:")
    for i, a in enumerate(ranking, 1):
        bar = '█' * int(suggested_weights.get(a, 0) * 30)
        print(f"    {i}. {a:<14}  weight: {suggested_weights.get(a, 0):.4f}  {bar}")

    print("\n  Final ranking is a user override.")
    print("  After you swap / move items, the final order no longer needs to match the quiz weights.")

    print("\n  Commands:")
    print("    swap X Y  — swap positions X and Y  (e.g. 'swap 2 4')")
    print("    move X Y  — move item at X to position Y, shift others down")
    print("    ENTER     — confirm ranking as shown")

    while True:
        raw = input("\n  > ").strip().lower()
        if not raw:
            break

        parts = raw.split()
        if len(parts) == 3 and parts[0] == 'swap':
            try:
                i, j = int(parts[1]) - 1, int(parts[2]) - 1
                if 0 <= i < len(ranking) and 0 <= j < len(ranking):
                    ranking[i], ranking[j] = ranking[j], ranking[i]
                else:
                    print("  Invalid positions.")
                    continue
            except ValueError:
                print("  Format: swap X Y")
                continue

        elif len(parts) == 3 and parts[0] == 'move':
            try:
                src, dst = int(parts[1]) - 1, int(parts[2]) - 1
                if 0 <= src < len(ranking) and 0 <= dst < len(ranking):
                    item = ranking.pop(src)
                    ranking.insert(dst, item)
                else:
                    print("  Invalid positions.")
                    continue
            except ValueError:
                print("  Format: move X Y")
                continue
        else:
            print("  Commands: 'swap X Y', 'move X Y', or ENTER to confirm")
            continue

        print("\n  Current final user-confirmed ranking:")
        for i, a in enumerate(ranking, 1):
            print(f"    {i}. {a}")

    return ranking


final_ranking = user_adjust_ranking(list(post_tb_rank), normalised_weights)

print_divider("Final User-Confirmed Ranking")
for i, a in enumerate(final_ranking, 1):
    print(f"  {i}. {a}")

print(f"\n  ✓ Final ranking confirmed: {final_ranking}")



───────────── STAGE 3: Confirm / Adjust Ranking ─────────────

  Suggested amenity ranking based on your quiz answers:
    1. bus             weight: 0.2049  ██████
    2. train           weight: 0.1972  █████
    3. school          weight: 0.1672  █████
    4. supermarket     weight: 0.1640  ████
    5. polyclinic      weight: 0.1366  ████
    6. mall            weight: 0.0911  ██
    7. hawker          weight: 0.0390  █

  Final ranking is a user override.
  After you swap / move items, the final order no longer needs to match the quiz weights.

  Commands:
    swap X Y  — swap positions X and Y  (e.g. 'swap 2 4')
    move X Y  — move item at X to position Y, shift others down
    ENTER     — confirm ranking as shown

  > 

──────────────── Final User-Confirmed Ranking ────────────────
  1. bus
  2. train
  3. school
  4. supermarket
  5. polyclinic
  6. mall
  7. hawker

  ✓ Final ranking confirmed: ['bus', 'train', 'school', 'supermarket', 'polyclinic', 'mall', 'hawker']


## ⚖️ Cell 12 — Rank-Sum Weight Calculation
Converts the final ranking into Rank-Sum weights.  
**Formula:** weight(rank i) = (n − i + 1) / sum(1..n)


In [ ]:
def compute_and_print_weights(ranking):
    weights = rank_sum_weights(ranking)
    n       = len(ranking)
    total   = int(n * (n + 1) / 2)

    print_divider("Rank-Sum Weight Calculation")
    print(f"\n  n = {n} active amenities")
    print(f"  Denominator = 1+2+...+{n} = {total}")
    print(f"\n  {'Rank':<6} {'Amenity':<14} {'Points':>8} {'Weight':>10}  {'Bar'}")
    print(f"  {'─'*6} {'─'*14} {'─'*8} {'─'*10}  {'─'*20}")
    for i, a in enumerate(ranking, 1):
        pts = n - (i-1)
        bar = '█' * int(weights[a] * 40)
        print(f"  {i:<6} {a:<14} {pts:>8} {weights[a]:>10.4f}  {bar}")

    print(f"\n  Sum of weights = {sum(weights.values()):.4f}  (should be 1.0000)")
    return weights


weights = compute_and_print_weights(final_ranking)



──────────────── Rank-Sum Weight Calculation ────────────────

  n = 7 active amenities
  Denominator = 1+2+...+7 = 28

  Rank   Amenity          Points     Weight  Bar
  ────── ────────────── ──────── ──────────  ────────────────────
  1      bus                   7     0.2500  ██████████
  2      train                 6     0.2143  ████████
  3      school                5     0.1786  ███████
  4      supermarket           4     0.1429  █████
  5      polyclinic            3     0.1071  ████
  6      mall                  2     0.0714  ██
  7      hawker                1     0.0357  █

  Sum of weights = 1.0000  (should be 1.0000)


## 🎚️ Cell 13 — Value vs Amenity Balance (Alpha)


In [ ]:
def choose_alpha():
    print_divider("Value vs Amenity Balance")
    print("\n  The final score = alpha × amenity_score + (1−alpha) × value_score")
    print("  Choose how much weight to give amenity fit vs price value:\n")

    choice = ask("How would you like to balance amenity fit vs price value?", {
        'fit':     'Prioritise amenity fit   (alpha = 0.90)',
        'balanced':'Balanced                 (alpha = 0.75)',
        'value':   'Prioritise value         (alpha = 0.60)',
    })
    alpha_map = {'fit': 0.90, 'balanced': 0.75, 'value': 0.60}
    alpha = alpha_map[choice]

    print(f"\n  → alpha = {alpha}")
    print(f"     Amenity component : {alpha:.0%} of final score")
    print(f"     Value component   : {1-alpha:.0%} of final score")
    return alpha


alpha = choose_alpha()



────────────────── Value vs Amenity Balance ──────────────────

  The final score = alpha × amenity_score + (1−alpha) × value_score
  Choose how much weight to give amenity fit vs price value:


How would you like to balance amenity fit vs price value?
  [A] Prioritise amenity fit   (alpha = 0.90)
  [B] Balanced                 (alpha = 0.75)
  [C] Prioritise value         (alpha = 0.60)
  Your choice: A
  → Prioritise amenity fit   (alpha = 0.90)

  → alpha = 0.9
     Amenity component : 90% of final score
     Value component   : 10% of final score


## 🏠 Cell 14 — Stage 3: Score Listings & Results
Scores all filtered listings using amenity accessibility (Haversine decay) + value score.  
Shortlists top 30, then returns top 5.


In [ ]:
def score_listings(filtered, amenity_df, weights, final_ranking, alpha):
    """
    Score each listing:
      amenity_score = sum(weight_i * accessibility_i) for active amenities
      value_score   = normalised pct_gap clipped to ±VALUE_CLIP
      final_score   = alpha * amenity_score + (1-alpha) * value_score
    """
    tau = TAU_DEFAULT
    print(f"\n  Using TAU = {tau}m")

    matched = match_to_amenity_table(filtered, amenity_df)
    records = []

    for i, (_, listing) in enumerate(filtered.iterrows()):
        if matched[i] is None:
            continue
        amenity_row   = amenity_df.iloc[matched[i]]
        acc           = get_amenity_scores(amenity_row, tau)
        amenity_score = sum(weights.get(a, 0) * acc.get(a, 0) for a in final_ranking)

        # Value score: clip pct_gap to ±VALUE_CLIP, normalise to 0-1
        pct_gap     = listing['value_delta_pct'] / 100
        clipped     = float(np.clip(pct_gap, -VALUE_CLIP, VALUE_CLIP))
        value_score = (clipped + VALUE_CLIP) / (2 * VALUE_CLIP)
        final_score = alpha * amenity_score + (1 - alpha) * value_score

        records.append({
            'address':          listing['full_address'],
            'town':             listing['planning_area'],
            'bedrooms':         listing['bedrooms'],
            'floor_area_sqft':  listing['floor_area_sqft'],
            'asking_price':     listing['asking_price'],
            'predicted_price':  listing['predicted_price'],
            'value_delta_pct':  listing['value_delta_pct'],
            'value_delta_abs':  listing['value_delta_abs'],
            'amenity_score':    round(amenity_score, 4),
            'value_score':      round(value_score, 4),
            'final_score':      round(final_score, 4),
            **{f'acc_{a}': round(acc.get(a, 0), 4) for a in final_ranking},
        })

    return pd.DataFrame(records).sort_values('final_score', ascending=False).reset_index(drop=True)


def print_results(scored, final_ranking, alpha, is_clustered=False):
    shortlist = scored.head(SHORTLIST_N)
    top_n     = shortlist.head(TOP_N)

    print_divider("RESULTS")
    print(f"\n  Shortlisted top {min(SHORTLIST_N, len(scored))} → returning top {min(TOP_N, len(scored))}")
    print(f"  Scoring: {alpha}×amenity + {round(1-alpha,2)}×value  |  TAU = {TAU_DEFAULT}m")

    if is_clustered:
        print("\n  📍 Listings are geographically clustered — amenity scores are similar.")
        print("     Price value is the main differentiator in these results.")

    print()
    for rank, (_, row) in enumerate(top_n.iterrows(), 1):
        delta_pct = row['value_delta_pct']
        delta_abs = row['value_delta_abs']

        # Value label
        # value_delta_pct = (predicted - asking) / predicted * 100
        # positive → predicted > asking → listing is undervalued (good deal) 🟢
        # negative → predicted < asking → listing is overvalued (asking too much) 🔴
        if delta_pct > 5:
            val_emoji = "🟢"; val_label = f"{delta_pct:.1f}% undervalued  (${abs(delta_abs):,} below predicted)"
        elif delta_pct > -5:
            val_emoji = "🟡"; val_label = f"~at market  ({delta_pct:+.1f}%)"
        else:
            val_emoji = "🔴"; val_label = f"{abs(delta_pct):.1f}% overvalued  (${abs(delta_abs):,} above predicted)"

        # Value rank label for clustered results
        if is_clustered:
            val_ranks = scored['value_score'].rank(ascending=False).astype(int)
            vrank = val_ranks.iloc[rank-1]
            val_label += f"  — #{vrank} value in pool"

        print(f"  ── #{rank} {'─'*52}")
        print(f"  {row['address']}")
        print(f"  {row['town']} | {int(row['bedrooms'])} bed | {int(row['floor_area_sqft']):,} sqft")
        print(f"  Asking: ${int(row['asking_price']):,}   Predicted: ${int(row['predicted_price']):,}   {val_emoji} {val_label}")
        print(f"  Amenity score: {row['amenity_score']:.4f}  |  Value score: {row['value_score']:.4f}  |  Final: {row['final_score']:.4f}")
        print(f"  Accessibility breakdown (TAU={TAU_DEFAULT}m):")
        for a in final_ranking:
            col = f'acc_{a}'
            if col in row:
                bar = '█' * int(row[col] * 25)
                print(f"    {a:<14} {row[col]:.4f}  {bar}")
        print()


# ── Run scoring ───────────────────────────────────────────────────
print(f"  Scoring {len(filtered)} filtered listings...")
scored = score_listings(filtered, amenity_df, weights, final_ranking, alpha)
print_results(scored, final_ranking, alpha, is_clustered=constraints.get('is_clustered', False))


  Scoring 417 filtered listings...

  Using TAU = 500.0m

────────────────────────── RESULTS ──────────────────────────

  Shortlisted top 30 → returning top 5
  Scoring: 0.9×amenity + 0.1×value  |  TAU = 500.0m

  ── #1 ────────────────────────────────────────────────────
  441C Fernvale Road
  SENGKANG | 3 bed | 1,001 sqft
  Asking: $630,000   Predicted: $648,000   🟡 ~at market  (+2.8%)
  Amenity score: 0.5272  |  Value score: 0.6400  |  Final: 0.5385
  Accessibility breakdown (TAU=500.0m):
    bus            0.7702  ███████████████████
    train          0.4442  ███████████
    school         0.6562  ████████████████
    supermarket    0.5536  █████████████
    polyclinic     0.0058  
    mall           0.4880  ████████████
    hawker         0.2181  █████

  ── #2 ────────────────────────────────────────────────────
  320A Anchorvale Drive
  SENGKANG | 3 bed | 947 sqft
  Asking: $599,000   Predicted: $643,000   🟢 6.8% undervalued  ($44,000 below predicted)
  Amenity score: 0.4948  

## 🚧 TODO — Planned Integrations

### 1. Replace Simulated Predicted Prices with Real Model Output
**Location:** Cell 4 `load_data()` — the simulation block clearly marked with `# Simulate predicted price`

**Current behaviour:** Predicted prices are generated by adding Gaussian noise (~±5%) to asking prices with a slight downward bias. This is for mockup/demo purposes only.

**What to replace it with:**  
Option A — Add predicted price as a column upstream in the scrape/cleaning pipeline, so by the time the listings file is loaded here, `predicted_price` already exists as a column. Then delete the simulation block entirely.

Option B — Call the price model inline here:
```python
# Replace simulation block with:
from your_model_module import predict_prices
listings['predicted_price'] = predict_prices(listings)
```

**What stays the same regardless:** The `value_delta_pct` and `value_delta_abs` derivation lines below the simulation block — these just compute from whatever `predicted_price` ends up being.

**Note:** Once the real model is in, consider adding a confidence column (e.g. prediction interval width) and passing it through to Cell 16 to suppress or flag the value score for low-confidence predictions. See Discussion Point 6 in the README.

---

### 2. OneMap Routing API — Walking Time Refinement (Stage 3B)
**Location:** Cell 16 `score_listings()` — currently uses Haversine distance for both Stage 3A shortlisting and final scoring.

**What Stage 3B should do:**  
After shortlisting top 30 via Haversine, call the OneMap routing API to get actual walking times (in minutes) for each shortlisted listing × each active amenity × top 3 nearest locations. Then recompute accessibility scores using `TAU_WALKING` instead of `TAU_HAVERSINE` and re-rank the shortlist to return the final top 5.

**API reference:** [https://www.onemap.gov.sg/apidocs/](https://www.onemap.gov.sg/apidocs/)  
Endpoint: `routing` with `routeType=walk`

**Rough call volume:** 30 listings × 7 amenities × 3 nearest = 630 API calls per user session max. In practice fewer since only active amenities are scored.

**Where to slot in:**
```python
# In score_listings(), after shortlist = scored.head(SHORTLIST_N):

# --- STAGE 3B: Walking time refinement (TODO) ---
# For each listing in shortlist:
#   For each active amenity:
#     Call OneMap routing API for top 3 nearest locations
#     Get walking_time_minutes for each
#   Recompute accessibility using exp_decay(walking_times, TAU_WALKING)
#   Recompute amenity_score with same weights
#   Re-sort shortlist by new final_score
#   Return top TOP_N
# ------------------------------------------------
```

**TAU_WALKING values to use** (already defined in Cell 3 config — just uncomment when ready):
```python
TAU_WALKING = 8.0   # minutes — score drops to 37% at 8 min walk
```

## 📋 Cell 15 — Optional: View Full Scored Table
Run this cell to see the complete scored shortlist as a dataframe.


In [ ]:
# Top 30 shortlist with all scores
display(scored.head(SHORTLIST_N)[[
    'address','town','bedrooms','floor_area_sqft',
    'asking_price','predicted_price','value_delta_pct','value_delta_abs',
    'amenity_score','value_score','final_score'
]])


,address,town,bedrooms,floor_area_sqft,asking_price,predicted_price,value_delta_pct,value_delta_abs,amenity_score,value_score,final_score
0,441C Fernvale Road,SENGKANG,3,1001,630000,648000.0,2.8,18000,0.5272,0.640,0.5385
1,320A Anchorvale Drive,SENGKANG,3,947,599000,643000.0,6.8,44000,0.4948,0.840,0.5293
2,615 Bukit Panjang Ring Road,BUKIT PANJANG,3,1184,648888,691000.0,6.1,42000,0.4967,0.805,0.5275
3,172C Edgedale Plains,PUNGGOL,3,1184,680000,685000.0,0.7,5000,0.5160,0.535,0.5179
4,172 Gangsa Road,BUKIT PANJANG,3,1087,700000,737000.0,5.0,37000,0.4906,0.750,0.5165
5,178 Edgefield Plains,PUNGGOL,2,1184,638000,645000.0,1.1,7000,0.5112,0.555,0.5156
6,612D Punggol Drive,PUNGGOL,3,1001,670000,682000.0,1.8,12000,0.5012,0.590,0.5101
7,205 Choa Chu Kang Central,CHOA CHU KANG,3,1313,660000,689000.0,4.2,29000,0.4834,0.710,0.5060
8,175A Punggol Field,PUNGGOL,3,1184,698000,685000.0,-1.9,-13000,0.5142,0.405,0.5032
9,621 Senja Road,BUKIT PANJANG,3,980,550000,551000.0,0.2,1000,0.5010,0.510,0.5019
